<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/sample-answers/data_wrangling_assignment_answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Assignment: Building a Modular Data Sanitization & Exploration Engine**

### **Background**

In real-world data science, **80% of the work is spent cleaning and exploring data**. Most of this work is repetitive: checking for nulls, identifying outliers, and visualizing distributions. Your task is to move away from "spaghetti code" (loose script lines) and build a **Reusable Python Class** that can be imported into any Google Colab environment to automate these tasks.

### **The Objective**

Develop a Python class named `DataInspector` that serves as an end-to-end tool for CSV data ingestion, cleaning, and interactive visualization.

### **Technical Requirements**

#### **1. Data Ingestion & Sanitization**

* Implement a method to handle local file uploads in Google Colab.
* The engine must automatically recognize and convert common "garbage strings" (e.g., `'?'`, `'n/a'`, `'NULL'`) into actual `NaN` values.
* The class must force-convert columns to their most appropriate numeric types where possible.

#### **2. Structural Analysis & Cleaning**

* **Summary:** Provide a method that identifies the count of categorical vs. numerical columns and displays a data preview.
* **Missing Data:** Create a diagnostic tool that filters and displays **only** the rows containing missing values.
* **Targeted Deletion:** Implement a row-deletion method that accepts a list of specific indices from the user to prune the dataset.
* **Outlier Management:** Develop an **IQR-based** outlier detection system. It must allow the user to specify which columns to check and provide an optional parameter to automatically delete those rows.

#### **3. Advanced Interactive Visualization**

All graphics must be implemented using **Plotly** to allow for interactivity (hovering, zooming).

* **Univariate Analysis:** For any given numeric column, generate a 3-panel subplot containing a **Horizontal Violin/Box plot**, a **Scatter plot**, and a **Histogram**.
* **Categorical Analysis:** Create bar charts that automatically calculate and display the **percentage frequency** of each category.
* **Bivariate Analysis:** Create a "smart" relationship plotter that detects column types and automatically chooses between a **Regression Plot** (Num-Num), **Box Plot** (Num-Cat), or **Grouped Bar** (Cat-Cat).
* **Multivariate Correlation:** Generate a **Pearson Correlation Heatmap** with text annotations to show the linear relationships between all numerical features.

### **Submission Criteria**

* The code must be object-oriented (encapsulated within a Class).
* Every method must include descriptive **Docstrings**.
* The final output should be tested against a "dirty" dataset (like the Titanic or Cafe Sales set) to demonstrate that outliers and missing values are handled correctly.

---

**Would you like me to create a "Grading Rubric" for this assignment so you can see how a professor might score your code?**

#Sample Answer

In [ ]:
import pandas as pd
import numpy as np
import io
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files

class DataInspector:
    """
    A comprehensive data cleaning and exploration tool for Google Colab.
    Provides interactive visualizations using Plotly and robust data sanitization.
    """

    def __init__(self):
        self.df = None

    def upload_data(self):
        """
        Prompts user to upload a CSV, handles common null strings,
        and attempts to auto-convert columns to their correct numeric types.
        """
        uploaded = files.upload()
        if not uploaded:
            return print("No file uploaded.")

        file_name = list(uploaded.keys())[0]
        self.df = pd.read_csv(io.BytesIO(uploaded[file_name]),
                              na_values=['?', 'n/a', 'N/A', 'NULL', 'null', ' '])

        for col in self.df.columns:
            self.df[col] = pd.to_numeric(self.df[col], errors='ignore')

        print(f"\n✅ File '{file_name}' loaded and types sanitized!")

    def get_summary(self):
        """
        Prints data dimensions and column type breakdown.
        Displays the first 20 rows of the dataframe.
        """
        if self.df is None: return print("Error: No data loaded.")

        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()

        print(f"--- Data Summary ---")
        print(f"Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}")
        print(f"Numerical ({len(num_cols)}): {num_cols}")
        print(f"Categorical ({len(cat_cols)}): {cat_cols}")
        display(self.df.head(20))

    def show_missing_data(self):
        """
        Filters the dataframe to show only rows containing at least one missing (NaN) value.
        """
        if self.df is None: return
        missing_mask = self.df.isnull().any(axis=1) | (self.df == "").any(axis=1)
        missing_rows = self.df[missing_mask]

        if missing_rows.empty:
            print("✨ No missing data found!")
        else:
            print(f"🔍 Found {len(missing_rows)} rows with missing values:")
            display(missing_rows)

    def delete_rows(self):
        """
        Deletes rows based on a comma-separated list of indices provided via user input.
        """
        if self.df is None: return
        try:
            user_input = input("Enter row indices to delete (e.g., 1, 3, 15): ")
            indices_to_drop = [int(i.strip()) for i in user_input.split(',') if i.strip().isdigit()]

            existing_indices = [i for i in indices_to_drop if i in self.df.index]
            self.df = self.df.drop(index=existing_indices).reset_index(drop=True)
            print(f"🗑️ Deleted {len(existing_indices)} rows. New count: {len(self.df)}")
        except Exception as e:
            print(f"❌ Error: {e}")

    def handle_missing_values(self, columns=None, strategy='median', fill_value=None):
        """
        Imputes missing values in specified columns to preserve data rows.

        Parameters:
        - columns: List of strings. If None, applies to all columns with NaNs.
        - strategy: 'mean', 'median', 'mode', or 'constant'.
        - fill_value: Used only if strategy is 'constant'.
        """
        if self.df is None: return
        target_cols = columns if columns else self.df.columns[self.df.isnull().any()].tolist()

        for col in target_cols:
            if strategy == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].mean())
            elif strategy == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
                self.df[col] = self.df[col].fillna(self.df[col].median())
            elif strategy == 'mode':
                self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
            elif strategy == 'constant':
                self.df[col] = self.df[col].fillna(fill_value)

        print(f"🛠️ Imputation complete using '{strategy}' strategy for: {target_cols}")

    def remove_duplicates(self):
        """
        Identifies and removes exact duplicate rows from the dataframe to prevent statistical bias.
        """
        if self.df is None: return
        initial_count = len(self.df)
        self.df = self.df.drop_duplicates().reset_index(drop=True)
        dropped = initial_count - len(self.df)
        print(f"✨ Removed {dropped} duplicate rows. New row count: {len(self.df)}")


    def export_cleaned_data(self, filename='cleaned_data.csv'):
        """
        Converts the current state of the dataframe to a CSV file and
        triggers a browser download in the Google Colab environment.
        """
        if self.df is None: return
        self.df.to_csv(filename, index=False)
        files.download(filename)
        print(f"💾 '{filename}' has been generated and download triggered.")

    def column_details(self):
        """
        Iterates through all columns to show numeric ranges or categorical unique value counts.
        """
        if self.df is None: return
        for col in self.df.columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                print(f"🔹 {col} (Numeric): Range [{self.df[col].min()} to {self.df[col].max()}]")
            else:
                print(f"🔸 {col} (Categorical): {self.df[col].nunique()} unique values")

    def get_categorical_summary(self):
        """
        Generates a detailed statistical summary for categorical columns,
        including unique counts, the most frequent value (Mode), and its frequency.
        """
        if self.df is None: return
        cat_df = self.df.select_dtypes(exclude=[np.number])
        if cat_df.empty:
            return print("No categorical columns found.")

        summary = cat_df.describe().T[['unique', 'top', 'freq']]
        print("--- Categorical Deep Dive ---")
        display(summary)

    def plot_numerical(self, column_names):
            """
            Generates interactive Horizontal Violin, Scatter, and Histogram plots.
            Swapping the axis for Violin/Box plots to improve horizontal comparison.
            """
            if self.df is None: return
            if isinstance(column_names, str): column_names = [column_names]

            valid_cols = [c for c in column_names if c in self.df.columns and pd.api.types.is_numeric_dtype(self.df[c])]

            for col in valid_cols:
                # Create a 1x3 grid for each numeric column
                fig = make_subplots(
                    rows=1, cols=3,
                    subplot_titles=(f"Horizontal Violin/Box: {col}", f"Scatter Plot: {col}", f"Distribution: {col}")
                )

                # --- 1. Horizontal Violin + Box (Changed y= to x=) ---
                fig.add_trace(
                    go.Violin(x=self.df[col], box_visible=True, meanline_visible=True,
                            name=col, orientation='h', line_color='lightseagreen'),
                    row=1, col=1
                )

                # --- 2. Scatter Plot (Index vs Value) ---
                fig.add_trace(
                    go.Scatter(y=self.df[col], mode='markers',
                            marker=dict(opacity=0.5, color='royalblue'), name=col),
                    row=1, col=2
                )

                # --- 3. Histogram ---
                fig.add_trace(
                    go.Histogram(x=self.df[col], name=col, marker_color='indianred'),
                    row=1, col=3
                )

                # Update layout for a polished look
                fig.update_layout(
                    height=450,
                    title_text=f"<b>Statistical Analysis: {col}</b>",
                    showlegend=False,
                    template="plotly_white"
                )

                # Update axes labels for clarity
                fig.update_xaxes(title_text="Value", row=1, col=1)
                fig.update_yaxes(title_text="Value", row=1, col=2)
                fig.update_xaxes(title_text="Value", row=1, col=3)

                fig.show()

    def plot_categorical(self, column_names):
        """
        Generates interactive Bar charts for categorical columns showing counts and percentages.
        """
        if self.df is None: return
        if isinstance(column_names, str): column_names = [column_names]

        for col in column_names:
            counts = self.df[col].value_counts().reset_index()
            counts.columns = [col, 'count']
            counts['percentage'] = (counts['count'] / counts['count'].sum() * 100).round(1).astype(str) + '%'

            fig = px.bar(counts, x=col, y='count', text='percentage',
                         title=f"Frequency: {col}", color=col, color_discrete_sequence=px.colors.qualitative.Pastel)
            fig.show()

    def handle_outliers(self, columns=None, find_and_delete=False):
        """
        Flags outliers using IQR logic.
        Optionally deletes the flagged rows and updates the class instance.
        """
        if self.df is None: return
        target_cols = columns if columns else self.df.select_dtypes(include=[np.number]).columns.tolist()
        all_outliers = set()

        for col in target_cols:
            Q1, Q3 = self.df[col].quantile(0.25), self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = self.df[(self.df[col] < (Q1 - 1.5 * IQR)) | (self.df[col] > (Q3 + 1.5 * IQR))]
            all_outliers.update(outliers.index.tolist())
            print(f"🚨 {col}: Found {len(outliers)} outliers.")

        if all_outliers:
            display(self.df.loc[list(all_outliers)])
            if find_and_delete:
                self.df = self.df.drop(index=list(all_outliers)).reset_index(drop=True)
                print(f"🗑️ Deleted {len(all_outliers)} outlier rows.")

    def plot_relationship(self, col1, col2):
        """
        Intelligently selects the best interactive plot based on column types:
        - Num vs Num: Scatter with Trendline
        - Cat vs Num: Box plot with data points
        - Cat vs Cat: Grouped bar chart
        """
        if self.df is None: return
        is_num1, is_num2 = pd.api.types.is_numeric_dtype(self.df[col1]), pd.api.types.is_numeric_dtype(self.df[col2])

        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Correlation: {col1} vs {col2}")
        elif is_num1 != is_num2:
            num, cat = (col1, col2) if is_num1 else (col2, col1)
            fig = px.box(self.df, x=cat, y=num, points="all", color=cat, title=f"Distribution of {num} by {cat}")
        else:
            fig = px.histogram(self.df, x=col1, color=col2, barmode="group", title=f"Relationship: {col1} vs {col2}")

        fig.show()

    def plot_correlation_heatmap(self):
        """
        Displays an interactive Heatmap of the Pearson Correlation matrix
        for all numeric features in the dataset.
        """
        if self.df is None: return
        corr = self.df.select_dtypes(include=[np.number]).corr()
        fig = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale='RdBu_r',
                        title="Pearson Correlation Heatmap")
        fig.show()


# Initialize the interactive inspector
inspector = DataInspector()

In [ ]:
# Initialize the inspector
inspector = DataInspector()

In [ ]:
# Initialize and Upload
inspector.upload_data()

In [ ]:
# Direct URL to the Titanic CSV on GitHub
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)

# Now you can jump straight to the methods!
inspector.get_summary()

In [ ]:
inspector.column_details()

In [ ]:
inspector.get_categorical_summary()

In [ ]:
inspector.show_missing_data()

In [ ]:
inspector.plot_numerical(column_names=['Age','SibSp'])

In [ ]:
inspector.handle_outliers(columns=['Age'],find_and_delete=True)
# Or explicitly: inspector.handle_outliers(find_and_delete=False)

In [ ]:
inspector.plot_categorical(column_names=['Survived', 'Pclass', 'Sex', 'Embarked'])

In [ ]:
inspector.plot_relationship('Survived','Age')

In [ ]:
inspector.plot_correlation_heatmap()

In [ ]:
inspector.delete_rows()

In [ ]:
inspector.df